# Семинар 10. Предобработка данных: выбросы и пропуски

Ноутбук занятия: материал разбирается вместе с преподавателем, ячейки выполняются сверху вниз.

- Переименуйте файл в формате `Группа-Фамилия-Имя-seminar-10.ipynb`, например `2MP9-Ivanov-Ivan-seminar-10.ipynb`.
- Ноутбук читает файлы `datasets/legal/dtp-bryansk.csv` и `datasets/legal/dtp-bryansk-participants.csv` из репозитория курса: путь в ячейке задан от папки `notebooks/`, поэтому папки `notebooks/` и `datasets/` должны лежать рядом, как в репозитории. Файлов ноутбук не пишет.
- Упражнения к занятию лежат в папке `exercises/`: листок `seminar-10-tasks.md` с формулировками и ноутбук `seminar-10-tasks.ipynb` с заготовками и тестами; они решаются на тех же данных, но по Московской области. Ответы вводятся в тест Moodle «Семинар 10».
- Шпаргалка по функциям: `reference/pandas.md`, раздел «Выбросы и аномалии». Этап 2 проекта, `project/02-eda.md`, сдаётся на семинаре 11; его раздел «Выявление аномалий и проблем» собирается из сегодняшних приёмов, разбор — в разделе 7.

Что сегодня. С семинара 4 мы считаем средние, не спрашивая, из каких значений они сложены. Сегодняшний вопрос стоит до любого среднего: что делать с крайними значениями и пропусками. Три вопроса к крайнему значению; три правила, которые находят выбросы, и случаи, где правила ломаются; пять решений и цена каждого; пропуски со смыслом; повторы и логические проверки; журнал решений для этапа 2.

## 1. Ошибка, редкость или код

Значение, которое резко отличается от остальных, называют выбросом. Одни выбросы — ошибки ввода, другие — редкие, но настоящие события, третьи — служебные коды на месте чисел. Перепутать их дорого: удалить настоящее событие значит исказить статистику, оставить ошибку — тоже.

Подключение то же, что на семинаре 9, только без plotly и без папки для файлов. Таблица ДТП Брянской области загружается с датами, и первым делом — сводка `describe` по четырём счётчикам: участники, транспортные средства, погибшие, раненые.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style='whitegrid')                                         # оформление графиков по умолчанию
dtp = pd.read_csv('../datasets/legal/dtp-bryansk.csv', index_col='id', parse_dates=['datetime'])   # ДТП Брянской области
counts = ['participants_count', 'vehicles_count', 'dead_count', 'injured_count']
dtp[counts].describe().round(2)                                          # сводка четырёх счётчиков

Строка `75%`: у трёх четвертей ДТП не больше трёх участников, двух машин, ни одного погибшего и одного раненого. А в строке `max` — 20 участников, 7 машин, 17 погибших и 14 раненых. К любому крайнему значению три вопроса: возможно ли оно, повторяется ли, объяснимо ли. Начнём с погибших: `nlargest` с семинара 5 даёт три ДТП с наибольшим числом погибших.

In [ ]:
dtp.nlargest(3, 'dead_count')[['datetime', 'district', 'category'] + counts]   # самые тяжёлые ДТП

15 марта 2015 года в 23:20 в Почепском районе столкновение: 20 участников, 2 машины, 17 погибших. Утром того же дня там же — ещё одно столкновение, 9 участников и 4 погибших. Возможно ли 20 участников при двух машинах? Ответ в таблице участников: в каком транспорте ехал каждый участник этого ДТП.

In [ ]:
people = pd.read_csv('../datasets/legal/dtp-bryansk-participants.csv')   # участники ДТП
people[people['dtp_id'] == 28405]['vehicle_category'].value_counts()      # классы машин участников

19 участников ехали в транспорте класса «Одноэтажные длиной от 5 до 8 м» — так ГИБДД записывает небольшие автобусы; ещё один — в седельном тягаче. Значение возможное, объяснимое и, к несчастью, настоящее: это выброс, который удалять нельзя. Удалить строку значило бы стереть 17 погибших из статистики области.

Второй пример, сквозной для всего занятия, — возраст машины: год ДТП минус год выпуска. Возраст — свойство машины, а не человека, а в таблице участников машина повторяется в строке каждого, кто в ней ехал: автобус этого ДТП попал бы в подсчёт девятнадцать раз. Поэтому для возраста нужна таблица «одна строка — одна машина». Машину задают два столбца — номер ДТП и номер машины в нём, `vehicle_no`. Строки без машины — у пешеходов и других участников без транспорта — убирает `dropna(subset=['vehicle_no'])`, а `drop_duplicates` по двум столбцам ключа оставляет первую строку каждой машины. Это не очистка от повторов, а другая таблица: марка, класс и год выпуска у всех людей одной машины одинаковы, а роль, пол и здоровье в ней — данные одного из них, и о машине они ничего не говорят. Дата ДТП приходит к каждому участнику через `merge`, как на семинаре 6.

In [ ]:
people = people.merge(dtp[['datetime']], left_on='dtp_id', right_index=True)   # дата ДТП каждому участнику
vehicles = people.dropna(subset=['vehicle_no']).drop_duplicates(subset=['dtp_id', 'vehicle_no'])   # строка на машину
vehicles['vehicle_age'] = vehicles['datetime'].dt.year - vehicles['vehicle_year']   # возраст машины в годах
print(len(people), len(vehicles))
vehicles['vehicle_age'].describe().round(2)

Участников 24 717, машин 15 870. Возраст известен у 14 850 машин: у части машин не указан год выпуска. Медиана 10 лет, у трёх четвертей машин не больше 16, а максимум — 2 014 лет. Такое значение невозможно. Кто это? Самые маленькие годы выпуска даёт `nsmallest`.

In [ ]:
vehicles.nsmallest(12, 'vehicle_year')[['dtp_id', 'brand', 'vehicle_category', 'vehicle_year']]   # самые старые машины

Четыре велосипеда с годом выпуска 1 — невозможное значение, ошибка ввода: похоже, так записан неизвестный год. Дальше идут редкости: велосипед 1962 года, мотоциклы ИЖ, ВОСХОД и УРАЛ 1960-х, машины AUDI, ГАЗ и УАЗ 1970-х. Мотоцикл 1965 года выпуска — редкость, но он возможен; велосипед первого года нашей эры — нет. Особый случай — трактор марки УАЗ 1962 года: год возможен, но марка и класс спорят друг с другом, ведь УАЗ — марка автомобилей, а не тракторов. Одно из двух полей записано неверно, и по таблице не узнать какое, поэтому называть эту машину в отчёте «редким трактором» нельзя.

Третий вид крайних значений знаком по семинару 6: стаж 95, 96 и 99 лет — служебные коды, а не годы; ими займёмся в разделе 5.

Правило занятия: крайнее значение сначала смотрят, потом считают. Решение по четырём велосипедам — в разделе 4, а пока возраст берётся только там, где год выпуска не раньше 1900-го.

## 2. Форма распределения и правила выбросов

Как распределён возраст машины? Гистограмма и ящик с усами с семинара 8 на одной фигуре, а перед ними четыре числа: число значений, среднее, медиана и асимметрия. Асимметрию измеряет метод `skew()`: около нуля у симметричного распределения, больше нуля, когда длинный хвост справа, меньше нуля — когда слева. Подробно о форме распределения — на семинаре 11.

In [ ]:
age = vehicles.loc[vehicles['vehicle_year'] >= 1900, 'vehicle_age']      # возраст с настоящим годом выпуска
print(age.agg(['count', 'mean', 'median', 'skew']).round(2))
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(x=age, discrete=True, ax=axes[0])                           # гистограмма возраста
axes[0].set_title('Возраст машины')
axes[0].set_xlabel('Возраст, лет')
axes[0].set_ylabel('Число машин')
sns.boxplot(x=age, ax=axes[1])                                           # ящик с усами
axes[1].set_title('Возраст машины: ящик с усами')
axes[1].set_xlabel('Возраст, лет')
fig.tight_layout()

Гистограмма скошена: высокий горб слева и длинный пологий хвост вправо, к редким старым машинам. Среднее 11,49 больше медианы 10: хвост тянет среднее за собой, а медиана его не замечает. Асимметрия 0,93 — заметный правый хвост. Точки справа от уса — те самые редкие машины. Где начинается «редкая»? На это отвечают правила выбросов.

Первое правило — межквартильный размах, IQR. На семинаре 8 было сказано, что усы ящика тянутся не дальше полутора размахов ящика от его краёв; правило IQR ставит границы ровно там. Размах — разность третьего и первого квартилей, `iqr = q3 - q1`; нижняя граница `q1 - 1.5 * iqr`, верхняя `q3 + 1.5 * iqr`; всё за ними — выбросы. Квартили даёт `quantile` со списком долей, и два числа сразу раскладываются в две переменные.

In [ ]:
q1, q3 = age.quantile([0.25, 0.75])                                      # первый и третий квартили
iqr = q3 - q1                                                            # межквартильный размах
lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr                            # границы усов ящика
print(q1, q3, iqr, lower, upper)
(age > upper).sum()                                                      # машин старше верхней границы

Квартили 6 и 16 лет, размах 10, границы −9 и 31. Нижняя граница ниже нуля: для возраста она бессмысленна, правило работает только сверху. Ус ящика кончается не на границе, а на последнем значении внутри неё: левый ус стоит на нуле — моложе нуля машин нет, и до границы −9 ему не дотянуться. Старше 31 года 268 машин — это точки справа от уса.

Второе правило — три сигмы. z-оценка показывает, на сколько стандартных отклонений значение отстоит от среднего: `(s - s.mean()) / s.std()`. Выброс — значение, у которого z по модулю больше трёх; модуль берёт метод `abs()`, чтобы ловить и слишком большие, и слишком малые значения.

In [ ]:
z = (age - age.mean()) / age.std()                                       # z-оценка каждого значения
print(round(age.mean() + 3 * age.std(), 2))
(z.abs() > 3).sum()                                                      # выбросов по трём сигмам

Граница трёх сигм — 34,72 года, за ней 109 машин: в два с лишним раза меньше, чем по IQR. Стандартное отклонение само раздуто хвостом, и граница отъезжает дальше от середины.

Третье правило — процентили. 99-й процентиль — значение, ниже которого лежат 99 % данных; всё, что выше, считается выбросом. Три правила рядом — одним Series:

In [ ]:
p99 = age.quantile(0.99)                                                 # 99-й процентиль возраста
print(p99)
pd.Series({'IQR': (age > upper).sum(), 'три сигмы': (z.abs() > 3).sum(), '99-й процентиль': (age > p99).sum()})   # три правила

99-й процентиль — 33 года, старше него 142 машины: меньше одного процента, потому что машины ровно 33 лет стоят на самой границе и выбросом не считаются. Три правила — три числа: 268, 109 и 142. Ни одно не ошибка. Правило — договорённость, а не закон природы, и выбор правила — решение аналитика, которое записывается в отчёт.

## 3. Где правила ломаются

Правила придуманы для величины с одним горбом и плавными хвостами. Счётчики ДТП не такие: это целые числа с короткой шкалой, а у погибших, как показала сводка в разделе 1, нулём заняты не меньше трёх четвертей строк. Те же формулы работают сразу для всей таблицы: `quantile(0.25)` у таблицы даёт квартиль каждого столбца, а сравнение таблицы с Series идёт по именам столбцов. Квартили, число выбросов по IQR, граница и число выбросов по правилу трёх сигм для четырёх счётчиков:

In [ ]:
q1_c, q3_c = dtp[counts].quantile(0.25), dtp[counts].quantile(0.75)      # квартили каждого счётчика
iqr_c = q3_c - q1_c                                                       # размах каждого счётчика
z_c = (dtp[counts] - dtp[counts].mean()) / dtp[counts].std()             # z-оценки всех счётчиков
outside = (dtp[counts] < q1_c - 1.5 * iqr_c) | (dtp[counts] > q3_c + 1.5 * iqr_c)   # маска за границами IQR
pd.DataFrame({'q1': q1_c, 'q3': q3_c, 'выбросов IQR': outside.sum(),
              'граница трёх сигм': dtp[counts].mean() + 3 * dtp[counts].std(),
              'выбросов трёх сигм': (z_c.abs() > 3).sum()}).round(2)

У погибших оба квартиля нули: размах ноль, обе границы ноль, и «выбросом» по IQR становится каждое ДТП с погибшим, все 1 359. У раненых обе границы — единица: выбросы и ДТП без раненых, и ДТП с двумя ранеными, всего 2 849, больше четверти таблицы. Правило трёх сигм для участников режет на 6,01: «выброс» — любое ДТП с семью участниками и больше, таких 118. Счётчики с кучей нулей и целыми значениями правилам не по зубам. Для них порог задают по смыслу, например «больше восьми участников — проверить, был ли автобус», и записывают его в журнал.

Вернёмся к возрасту: выбросы с обеих сторон одной маской, `|` — «или», как на семинаре 4. Ошибка в следующей ячейке намеренная.

In [ ]:
age[age < lower | age > upper]                                           # выбросы с обеих сторон

`TypeError`: Python выполняет `|` раньше сравнений и читает маску как `age < (lower | age) > upper`, а побитовое «или» с дробным числом невозможно: границы из `quantile` дробные, даже когда на вид целые. Тот же `TypeError` дал бы и столбец целых чисел. `ValueError: The truth value of a Series is ambiguous` бывает, только когда целые и столбец, и граница: тогда `|` срабатывает, и Python спотыкается уже на двойном сравнении. Правило с семинара 4: каждое сравнение в маске — в скобках. Заодно посмотрим, что за машины за границей:

In [ ]:
outliers = age[(age < lower) | (age > upper)]                            # выбросы с обеих сторон
vehicles.loc[outliers.index, 'vehicle_category'].value_counts().head()   # классы самых старых машин

За границей не экзотика, а обычные легковые машины малых классов и мотоциклы: старые, но настоящие. Для отчёта это редкие значения, которые оставляют и помечают, — к решениям вернёмся в разделе 4.

При запасе времени: выброс бывает и в сводке. Число ДТП в каждый день — `value_counts` по дате без времени, `dt.date`, — и правило трёх сигм по дням. Рядом по годам — сколько пиковых дней пришлось на год и сколько в том году в таблице лёгких ДТП:

In [ ]:
daily = dtp['datetime'].dt.date.value_counts()                           # ДТП в каждый день
z_day = (daily - daily.mean()) / daily.std()                             # z-оценка каждого дня
peaks = daily[z_day > 3]                                                 # дни выше трёх сигм
print(round(daily.mean(), 2), daily.max(), daily.idxmax(), len(peaks))
light = dtp.loc[dtp['severity'] == 'Легкий', 'datetime'].dt.year         # годы лёгких ДТП
pd.DataFrame({'пиковых дней': pd.to_datetime(peaks.index).year.value_counts(),
              'лёгких ДТП': light.value_counts()}).fillna(0).astype(int).sort_index()   # пики и учёт по годам

В дни, когда ДТП были, их в среднем 2,93, максимум 12 — 16 июня 2017 года. Правило трёх сигм находит 39 дней, и все, кроме одного, — в 2015–2019 годах. Дело не в том, что ДТП тогда было больше: в те годы в таблицу попадали и лёгкие ДТП, от 546 до 629 в год, а в 2020-м — ни одного. Учёт лёгких ДТП сменился, об этом предупреждает описание данных, `datasets/legal/README.md`, раздел «Что стоит знать при анализе». Граница, посчитанная по всем годам сразу, отмечает не всплески, а годы прежнего учёта. Пик в сводке — не ошибка данных, а повод спросить, что случилось в тот день: погода, праздник, массовая авария; но сначала — одинаково ли считались дни, которые сравнивают.

## 4. Пять решений и их цена

Выброс найден — что с ним делать? Решений пять: оставить и пометить, исправить, удалить, срезать, заменить пропуском. Каждое по-своему меняет сводку. Начнём с невозможного: год выпуска 1. Настоящий год по таблице не восстановить, поэтому значение заменяют пропуском. Метод `replace` знаком по семинару 6; здесь он вызван с параметром `inplace=True`, который обещает «изменить на месте».

In [ ]:
vehicles['vehicle_year'].replace(1, np.nan, inplace=True)                # год 1 в пропуск
(vehicles['vehicle_year'] == 1).sum()                                    # машин с годом 1

Ловушка без ошибки: pandas выдал предупреждение `ChainedAssignmentError`, ячейка выполнилась, а год 1 остался у четырёх машин. `vehicles['vehicle_year']` — отдельный объект, и в pandas 3 изменение «на месте» в нём до таблицы не доходит. Правильно — присвоить результат обратно в столбец и пересчитать возраст. Прежний возраст сохраним для сравнения.

In [ ]:
raw_age = vehicles['vehicle_age']                                        # возраст до замены
vehicles['vehicle_year'] = vehicles['vehicle_year'].replace(1, np.nan)   # год 1 в пропуск
vehicles['vehicle_age'] = vehicles['datetime'].dt.year - vehicles['vehicle_year']   # возраст заново
print((vehicles['vehicle_year'] == 1).sum(), vehicles['vehicle_age'].max())

Года 1 больше нет, самой старой машине 62 года. Строки остались: класс машины и ДТП у этих велосипедов известны, и для этих столбцов строки годятся.

Исправить — значит вернуть настоящее значение, а не заменить его пропуском. Это возможно, только когда настоящее значение восстанавливается из самих данных. Год 1 не восстановить, а координаты иногда можно: на семинаре 8 за рамкой области нашлись ДТП, у которых широта и долгота записаны местами. Такую пару узнают по рамке: широта попадает в рамку долготы, а долгота — в рамку широты. Два столбца меняются местами одной строкой, как две переменные в Python: `a, b = b, a`.

In [ ]:
frame = dtp['lat'].between(51.8, 54.2) & dtp['lon'].between(31, 35.5)     # внутри рамки области
swapped = dtp['lat'].between(31, 35.5) & dtp['lon'].between(51.8, 54.2)   # широта и долгота местами
print((~frame).sum(), (dtp['lat'] == 0).sum(), swapped.sum())
dtp.loc[swapped, 'lat'], dtp.loc[swapped, 'lon'] = dtp.loc[swapped, 'lon'], dtp.loc[swapped, 'lat']   # поменять местами
(~(dtp['lat'].between(51.8, 54.2) & dtp['lon'].between(31, 35.5))).sum()   # вне рамки после исправления

За рамкой было 367 ДТП, из них 224 с нулевыми координатами. У 40 ДТП широта и долгота стояли местами; после перестановки эти ДТП внутри рамки, и за ней осталось 327. Исправление не теряет ни строки, ни значения. Нулевые координаты и точки других регионов так не восстановить: для них решение другое, к ним вернёмся в разделе 6.

Для редких, но настоящих значений решение — оставить и пометить: новый логический столбец «старше границы». Ловушка без ошибки: сравнение с пропуском всегда даёт `False`.

In [ ]:
vehicles['old_car'] = vehicles['vehicle_age'] > upper                    # старше верхней границы
normal = vehicles['vehicle_age'] <= upper                                # не старше границы
print(vehicles['old_car'].sum(), normal.sum(), vehicles['vehicle_age'].isna().sum(), len(vehicles))

268 старых и 14 578 не старых вместе дают 14 846 — столько возрастов известно, а машин 15 870. У 1 024 машин возраста нет, и они не попали ни в одну из двух масок: пропуск не больше и не меньше границы, он неизвестен. Фраза «старых машин 268 из 15 870» в отчёте была бы неверной: 268 из 14 846 с известным возрастом.

Остальные решения — в одной таблице: среднее, медиана и число значений после каждого. Метод `agg` со списком имён считает несколько статистик сразу. `clip(upper=...)` прижимает всё, что выше потолка, к потолку: это срез, или винзоризация. `where(условие)` оставляет значения там, где условие истинно, а остальные заменяет пропуском.

In [ ]:
age = vehicles['vehicle_age']                                            # возраст после замены
measures = ['mean', 'median', 'count']
pd.DataFrame({'как есть, с годом 1': raw_age.agg(measures),
              'год 1 в пропуск': age.agg(measures),
              'удалить старше границы': age[age <= upper].agg(measures),
              'срезать по 99-му процентилю': age.clip(upper=age.quantile(0.99)).agg(measures),
              'старше границы в пропуск': age.where(age <= upper).agg(measures)}).T.round(2)   # цена каждого решения

Четыре велосипеда из пятнадцати тысяч машин сдвигали среднее с 11,49 до 12,03. Удаление машин старше границы и замена их пропуском опускают среднее до 11,05, срез по 99-му процентилю — до 11,44. Медиана стоит на 10 при любом решении: она нечувствительна к хвосту, поэтому для скошенных данных её и приводят рядом со средним. Пометка `old_car` сводку не меняет совсем: она совпадает со строкой «год 1 в пропуск».

Удалить и заменить пропуском дают одну сводку, но разную таблицу: удаление уносит строку целиком, вместе с маркой, классом и ДТП, а пропуск оставляет строку для остальных столбцов. Поэтому удаляют только невозможное и только строки, которые больше ни на что не годны; здесь таких нет. Правило: решение записывается в журнал — что проверено, по какому правилу, сколько найдено, что сделано и почему.

## 5. Пропуски: не чем, а почему

Семинар 6 дал механику: `isna`, `fillna`, `dropna`. Теперь стратегия: прежде чем решать, чем заполнить пропуск, надо спросить, почему значения нет. Стаж с кодами 95, 96 и 99, заменёнными пропуском, и доля пропуска стажа по ролям: `crosstab` роли и маски пропусков с `normalize='index'` даёт доли внутри каждой роли.

In [ ]:
people['experience_clean'] = people['experience'].replace([95, 96, 99], np.nan)   # коды стажа пропусками
print(people['experience'].isin([95, 96, 99]).sum())
pd.crosstab(people['role'], people['experience_clean'].isna(), normalize='index').round(3)   # доля пропуска по ролям

Коды стоят в 1 403 строках участников. У всех ролей, кроме водителя, стаж пропущен в каждой строке: у пассажира, пешехода и велосипедиста стажа вождения нет по смыслу. Ловушка без ошибки — заполнить пропуски медианой по всей таблице:

In [ ]:
filled = people['experience_clean'].fillna(people['experience_clean'].median())   # пропуски медианой во всей таблице
filled[people['role'] == 'Пешеход'].value_counts()                       # «стаж» пешеходов

Каждый из 2 901 пешехода получил стаж 13 лет. Ошибки нет, а данные испорчены: стаж выдан людям, которые не водят, и средний стаж «участника» теперь считается и по ним. Пропуск со смыслом не заполняют, а отфильтровывают: стаж считается только по водителям.

У водителей стаж пропущен в 7,9 % строк, и здесь он быть должен. Три пути: оставить пропуск — статистики pandas его пропускают; заполнить медианой всех водителей; заполнить медианой своей группы, например пола. Медиану группы на каждой строке даёт `transform`: `groupby('gender')['experience_clean'].transform('median')` возвращает Series той же длины, что таблица, и `fillna` принимает его как есть. Для сравнения — заполнение нулём.

In [ ]:
drivers = people[people['role'] == 'Водитель']                           # только водители
experience = drivers['experience_clean']
by_gender = drivers.groupby('gender')['experience_clean'].transform('median')   # медиана стажа своего пола
pd.DataFrame({'оставить пропуск': experience.agg(measures),
              'медиана всех водителей': experience.fillna(experience.median()).agg(measures),
              'медиана своего пола': experience.fillna(by_gender).agg(measures),
              'ноль': experience.fillna(0).agg(measures)}).T.round(2)    # цена каждого заполнения

Без заполнения среднее 15,95 и медиана 13 по 14 028 водителям. Медиана всех водителей опускает среднее до 15,72: пропуски заменены числом меньше среднего. Медиана пола даёт 15,87 и сдвигает уже и медиану, с 13 до 14; значений 14 959, а не 15 229: у водителей без пола медиана группы не нашлась, и пропуск остался. Ноль опускает среднее до 14,69, а медиану до 12: нулевой стаж означает новичка, а не «неизвестно». Ни один путь не бесплатен, и выбранный записывается в журнал. Удалить водителей без стажа можно через `dropna(subset=['experience_clean'])` — только с `subset`, как на семинаре 6: без него строка пропадёт из-за пропуска в любом столбце.

Пропуск сам по себе бывает признаком. Индикатор пропуска — логический столбец, который сохраняет, где значения не было, даже после заполнения.

In [ ]:
drivers['experience_missing'] = drivers['experience_clean'].isna()       # пропуск стажа у водителя
pd.crosstab(drivers['experience_missing'], drivers['health'], normalize='index').round(3)   # здоровье водителей

Среди водителей без стажа погибли 10,5 %, ранены 54,4 %; среди водителей со стажем — 3,9 % и 29,9 %. Стаж чаще не записан у погибших и раненых водителей. После заполнения медианой эти водители стали бы неотличимы от остальных, а индикатор сохраняет, у кого стажа не было.

## 6. Повторы и логика

Семинар 6 нашёл в таблице участников строки, полностью повторяющие другие, и показал на одном ДТП, что это разные люди. Сколько их и кто они? `duplicated(keep=False)` помечает все строки каждой группы одинаковых, а `duplicated()` без параметров — все, кроме первой в группе: столько строк удалил бы `drop_duplicates`.

In [ ]:
repeated = people.duplicated(keep=False)                                 # строки в группах одинаковых
print(repeated.sum(), people.duplicated().sum())
people.loc[repeated, 'role'].value_counts()                              # роли в повторах

2 017 строк участников входят в группы одинаковых, `drop_duplicates` удалил бы 1 193 из них. Это только пассажиры и пешеходы: у пассажира нет своего номера, и два раненых пассажира одного пола из одной машины неотличимы. Ловушка без ошибки — удалить их как дубликаты. Что стало бы со сводкой по здоровью:

In [ ]:
health = pd.DataFrame({'все строки': people['health'].value_counts(),
                       'после drop_duplicates': people.drop_duplicates()['health'].value_counts()})   # сводка по здоровью
health['удалено'] = health['все строки'] - health['после drop_duplicates']
health

Из сводки пропали бы 1 110 раненых и 57 погибших: `drop_duplicates` удалил бы настоящих людей. Позиция курса: повтор не ошибка, если он объясним. Удаляют повторы только по явному ключу и только когда понятно, что такое строка: `drop_duplicates(subset=['dtp_id'])` оставляет по строке на ДТП, но это уже другая таблица, а не очистка.

Вторая часть раздела — логические проверки: правила, которые должны выполняться в каждой строке. Пострадавших не больше участников; машин не больше участников; машин не ноль; год выпуска не позже года ДТП; дата внутри 2015–2025 годов; координаты внутри рамки области с семинара 8. Каждое правило — маска нарушений, её сумма — число нарушений, и все проверки собираются в один Series. Тильда `~` переворачивает маску: «вне диапазона» — это «не внутри».

In [ ]:
checks = pd.Series({                                                     # нарушений каждого правила
    'пострадавших больше участников': (dtp['dead_count'] + dtp['injured_count'] > dtp['participants_count']).sum(),
    'машин больше участников': (dtp['vehicles_count'] > dtp['participants_count']).sum(),
    'машин нет': (dtp['vehicles_count'] == 0).sum(),
    'выпуск позже года ДТП': (vehicles['vehicle_year'] > vehicles['datetime'].dt.year).sum(),
    'дата вне 2015–2025 годов': (~dtp['datetime'].dt.year.between(2015, 2025)).sum(),
    'координаты вне рамки области': (~(dtp['lat'].between(51.8, 54.2) & dtp['lon'].between(31, 35.5))).sum(),
})
print(checks)
dtp.loc[dtp['vehicles_count'] > dtp['participants_count'], 'category'].value_counts().head(3)   # где машин больше

Четыре проверки из шести ничего не нашли. Проверка, которая ничего не нашла, — тоже результат, и её записывают: читатель отчёта узнаёт, что правило проверено, а не забыто. Машин больше участников в 175 ДТП, из них 118 столкновений и 25 наездов на стоящее ТС: у стоящей машины водителя за рулём нет, и в участники он не попадает. Записи участников неполные, но это не ошибка, а устройство данных. Координаты вне рамки у 327 ДТП — это уже после исправления переставленных в разделе 4. Остались нулевые координаты и точки других регионов: место ДТП по ним не восстановить, и решение для них — пропуск в `lat` и `lon`, строка остаётся для остальных столбцов.

## 7. Раздел «Аномалии» этапа 2 и журнал решений

Этап 2 проекта, `project/02-eda.md`, требует раздел «Выявление аномалий и проблем»: выбросы по числовым столбцам, логические проверки, дубликаты. Найти мало: по каждой находке нужно решение и его причина. Образец такого раздела — журнал решений по таблицам Брянской области, собранный из этого ноутбука. Столбцов четыре: что проверено, по какому правилу, что найдено и решение; в столбце «Решение» сначала что сделано, а после двоеточия — почему.

| Проверка | Правило | Найдено | Решение |
|---|---|---|---|
| Самое тяжёлое ДТП | три вопроса: возможно, повторяется, объяснимо | 17 погибших, 20 участников, 19 из них в автобусе | оставить: настоящее событие |
| Год выпуска | не раньше 1900 года | 4 велосипеда с годом 1 | заменить пропуском: настоящий год не восстановить, а строки нужны для остальных столбцов |
| Марка и класс машины | сочетание по смыслу | трактор марки УАЗ 1962 года | оставить: какое из двух полей неверно, не установить; в выводах на эту машину не опираться |
| Возраст машины | IQR по таблице машин, верхняя граница 31 год | 268 машин | оставить и пометить столбцом `old_car`: редкие, но настоящие |
| Возраст машины | три сигмы; 99-й процентиль | 109; 142 | только для сравнения правил: решение принято по IQR |
| Погибшие и раненые в ДТП | IQR | 1 359 и 2 849 ДТП | оставить: правило не годится для счётчиков с кучей нулей, порог задают по смыслу |
| Стаж | коды 95, 96, 99 | 1 403 строки участников | заменить пропуском в столбце `experience_clean`: код — не число лет |
| Стаж не у водителей | доля пропуска по ролям | пропуск в каждой строке | не заполнять: стажа нет по смыслу |
| Стаж водителей | доля пропуска | 7,9 % водителей | оставить пропуск и индикатор `experience_missing`: стаж чаще не записан у погибших и раненых, заполнение сдвинуло бы среднее |
| Повторы участников | `duplicated(keep=False)` | 2 017 строк участников | не удалять: неотличимые пассажиры и пешеходы — настоящие люди |
| Пострадавших больше участников; машин нет; выпуск позже года ДТП; дата вне периода | маски | 0 | записать: проверка без находок — тоже результат |
| Машин больше участников | маска | 175 ДТП | оставить: участники записаны не для каждой машины |
| Координаты местами | широта в рамке долготы, долгота в рамке широты | 40 ДТП | исправить, поменять обратно: настоящее значение восстанавливается |
| Координаты вне рамки | рамка области | 327 ДТП после исправления, из них 224 нулевые | заменить пропуском в `lat` и `lon`: место не восстановить, а строка нужна для остальных столбцов |

Памятка из пяти вопросов к выбросу:

1. Возможно ли значение? Невозможное — ошибка: исправить, если настоящее значение восстанавливается, иначе заменить пропуском.
2. Повторяется ли? Сотни одинаковых странных значений — код, а не случайность.
3. Объяснимо ли? Редкое объяснимое значение — настоящее: оставить и пометить.
4. Что меняет в сводке? Среднее, медиана и число значений до и после решения.
5. Что записано в журнал? Что проверено, по какому правилу, что найдено, что сделано и почему.

Необязательная задача сегодняшних упражнений — раздел «Аномалии» этапа 2 для своего датасета: таблица выбросов, логические проверки, повторы и журнал решений; он сдаётся вместе с этапом 2 на семинаре 11.

## Итоги

Крайнее значение сначала смотрят, потом считают: три вопроса — возможно ли, повторяется ли, объяснимо ли — отделяют ошибку от редкости и от кода. Свойство машины считают по таблице «одна строка — одна машина», которую даёт `drop_duplicates` по явному ключу, а не по строкам участников. Форму распределения показывают гистограмма и ящик, асимметрию измеряет `skew()`. Три правила выбросов: IQR — квартили через `quantile([0.25, 0.75])` и полтора размаха от них, а усы ящика доходят до последнего значения внутри этих границ; три сигмы — z-оценка по модулю больше трёх; процентили — всё за 1-м и 99-м. Правила дают разные числа и ломаются на счётчиках с кучей нулей и на сводках, где сменился учёт; маска из двух сравнений пишется со скобками. Пять решений — оставить и пометить, исправить, удалить, срезать через `clip`, заменить пропуском через `where` или `replace` — по-разному меняют среднее, а медиана стоит. Исправить — вернуть настоящее значение, как у переставленных координат; невозможное значение, которое не восстановить, заменяют пропуском. `inplace=True` у столбца в pandas 3 таблицу не меняет, результат присваивается обратно. Пропуск сначала объясняют, потом заполняют: у пешехода стажа нет, у водителя медиану группы даёт `groupby(...).transform('median')`; сравнение с пропуском всегда `False`. Повторы участников — неотличимые люди, и `drop_duplicates` их стёр бы; логические проверки — маски, и проверка без находок тоже идёт в журнал.

Переходите к упражнениям: откройте `exercises/seminar-10-tasks.ipynb`, формулировки в `exercises/seminar-10-tasks.md`; данные там по Московской области. Функции части Б — инструменты раздела «Аномалии» этапа 2: границы и маска выбросов, таблица выбросов, срез, пропуск вместо невозможного, заполнение медианой группы, доли пропусков, логические проверки; в части В из них собираются таблица «до и после», отчёт о повторах и очистка таблицы участников с журналом. Необязательная задача — раздел «Аномалии» этапа 2 для своего датасета, сдаётся вместе с этапом 2 на семинаре 11.